# WS-03 — Chuẩn bị gán nhãn yếu trên Google Colab

Notebook này chạy trên **Google Colab** cho bước 3 của quy trình Weak Supervision:

1. Kết nối Drive và lấy đúng branch `WS`.
2. Cài phụ thuộc Colab tối thiểu của RAD.
3. Đọc `configs/ws.yaml` và xác thực các đường dẫn SkinCAP đã được kiểm chứng.
4. Chỉ đọc và kiểm tra toàn bộ artifact đã chốt của WS_02 trên Drive.
5. Kiểm tra checkpoint text teacher, GPU và các điều kiện để bắt đầu gán nhãn yếu.

> **Ranh giới WS_02 / WS_03:** `propositions_review.csv`, `propositions_final.json`, `summary_per_disease.csv` và `meta.json` trong `WeakSup/propositions/v1/` là đầu vào **read-only** của WS_03. Notebook này không export, tạo lại, sửa hoặc ghi đè bất kỳ file WS_02 nào.
>
> **Phạm vi hiện tại:** repository mới có logic xuất propositions của bước 2. Runner gán nhãn yếu NLI / ConceptCLIP / LLM của bước 3 chưa được triển khai, nên notebook này không bịa lệnh chạy mô hình hoặc gọi API.
>
> Không lưu Hugging Face token, OpenAI API key hoặc Anthropic API key vào notebook hay YAML. Chỉ nhập chúng qua Colab Secrets / biến môi trường khi runner bước 3 đã sẵn sàng.

## 1. Thiết lập môi trường Google Colab

Kiểm tra phiên bản Python, GPU khả dụng và thư mục làm việc hiện tại. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU tương đương) trước khi chạy các phần liên quan đến mô hình.

In [ ]:
import os
import platform
import sys

try:
    import torch
except ImportError as error:
    raise RuntimeError("Notebook này phải được chạy trong Google Colab có PyTorch.") from error

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime: {torch.version.cuda}")
else:
    print("⚠️ Chưa có GPU. Bước gán nhãn mô hình nên chạy với GPU Colab.")

## 2. Cài đặt và nhập các thư viện

Cài bộ phụ thuộc tập trung cho SkinCAP trên Colab. Không dùng `requirements.txt` đầy đủ vì nó chứa nhiều gói phục vụ môi trường/nhánh khác, một số gói cần biên dịch và không phù hợp với Colab.

In [ ]:
# Các gói cơ bản cần cho các ô kiểm tra và đọc cấu hình.
# Các phụ thuộc SkinCAP/RAD sẽ được cài sau khi clone repository.
!pip -q install --no-cache-dir "PyYAML>=6.0" pandas numpy tqdm

import json
import subprocess
from pathlib import Path

import pandas as pd
import yaml

print("Đã import: pandas và PyYAML.")

## 3. Kết nối Google Drive và lấy branch `WS`

Drive cung cấp dữ liệu SkinCAP và nơi lưu artifact phiên bản. Repository được đặt tại `/content/RAD-colab`; ô này chỉ checkout branch `WS`, không tự tạo hay đổi các đường dẫn dữ liệu đã xác nhận.

In [ ]:
try:
    from google.colab import drive
except ImportError as error:
    raise RuntimeError("Hãy chạy notebook này trong Google Colab.") from error

drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/trisle0805/RAD-colab.git"
REPO_DIR = Path("/content/RAD-colab")
BRANCH = "WS"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
).strip()
assert current_branch == BRANCH, f"Expected branch {BRANCH!r}, got {current_branch!r}"
print(f"Repository: {REPO_DIR}")
print(f"Branch: {current_branch}")

In [ ]:
# The repository has now been cloned, so install its focused Colab dependency set.
requirements_file = REPO_DIR / "requirements" / "colab-skin.txt"
if not requirements_file.is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(requirements_file)],
    check=True,
)
print(f"Đã cài phụ thuộc từ: {requirements_file}")

## 4. Khai báo đường dẫn dữ liệu và kết quả

Tất cả các đường dẫn được lấy từ `configs/ws.yaml`. Artifact WS_02 trong `WeakSup/propositions/v1/` là **đầu vào chỉ đọc**. Notebook không sao chép, di chuyển, xuất lại hay chỉnh sửa file SkinCAP hoặc artifact WS_02.

In [ ]:
CONFIG_PATH = REPO_DIR / "configs" / "ws.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing WS config: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

# Repository paths are intentionally asserted rather than silently changed.
assert Path(cfg["repo_dir"]) == REPO_DIR, (
    f"configs/ws.yaml repo_dir must be {REPO_DIR}, got {cfg['repo_dir']}"
)

PATHS = {
    "train_csv": Path(cfg["skincap"]["train_csv"]),
    "val_csv": Path(cfg["skincap"]["val_csv"]),
    "test_csv": Path(cfg["skincap"]["test_csv"]),
    "image_root": Path(cfg["skincap"]["image_root"]),
    # Read-only artifacts produced by WS_02 on Drive.
    "propositions_dir": Path(cfg["propositions_dir"]),
    "propositions_review": Path(cfg["propositions_review"]),
    "propositions_final": Path(cfg["propositions_final"]),
    "propositions_summary": Path(cfg["propositions_summary"]),
    "propositions_meta": Path(cfg["propositions_meta"]),
    # Future Stage-03 inputs/outputs; their existence is checked separately.
    "teacher_text_model": Path(cfg["teacher_text_model"]),
    "weak_labels_dir": Path(cfg["weak_labels_dir"]),
}

print("Cấu hình đã nạp từ:", CONFIG_PATH)
for name, path in PATHS.items():
    print(f"- {name}: {path}")

In [ ]:
def require_file(label: str, path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{label} không tồn tại: {path}")
    return path


def require_directory(label: str, path: Path) -> Path:
    if not path.is_dir():
        raise NotADirectoryError(f"{label} không tồn tại hoặc không phải thư mục: {path}")
    return path


for split in ("train_csv", "val_csv", "test_csv"):
    require_file(split, PATHS[split])
require_directory("image_root", PATHS["image_root"])

# WS_03 requires the complete, already-approved WS_02 version on Drive.
require_directory("propositions_dir", PATHS["propositions_dir"])
for artifact in (
    "propositions_review",
    "propositions_final",
    "propositions_summary",
    "propositions_meta",
):
    require_file(artifact, PATHS[artifact])

# Check split schema and case counts without copying data.
split_frames = {}
for split in ("train", "val", "test"):
    csv_path = PATHS[f"{split}_csv"]
    frame = pd.read_csv(csv_path)
    if frame.empty:
        raise ValueError(f"{split} split is empty: {csv_path}")
    split_frames[split] = frame
    print(f"{split:>5}: {len(frame):,} rows | columns: {list(frame.columns)}")

image_count = sum(1 for path in PATHS["image_root"].iterdir() if path.is_file())
print(f"Images directly in image_root: {image_count:,}")
print("WS_02 artifacts: present and read-only.")

## 5. Chạy quy trình chính — đọc artifact đã chốt của WS_02

Bước 3 chỉ đọc `propositions_final.json` để lấy danh sách proposition, hypothesis và cờ `image_applicable`. `propositions_review.csv`, `summary_per_disease.csv` và `meta.json` được kiểm tra để bảo đảm bộ artifact WS_02 đầy đủ và truy vết được. Notebook không gọi `export_final_version` và không ghi vào thư mục `WeakSup/propositions/v1/`.

In [ ]:
propositions_dir = PATHS["propositions_dir"]
propositions_final = PATHS["propositions_final"]

# Read-only: WS_02 has already produced and approved this version on Drive.
with propositions_final.open("r", encoding="utf-8") as handle:
    propositions = json.load(handle)

if not isinstance(propositions, list) or not propositions:
    raise ValueError(f"Invalid or empty propositions file: {propositions_final}")

with PATHS["propositions_meta"].open("r", encoding="utf-8") as handle:
    propositions_meta = json.load(handle)

summary = pd.read_csv(PATHS["propositions_summary"])
if summary.empty:
    raise ValueError(f"Empty WS_02 summary: {PATHS['propositions_summary']}")

print(f"WS_02 directory (read-only): {propositions_dir}")
print(f"Observable propositions: {len(propositions):,}")
print(f"Diseases in WS_02 summary: {len(summary):,}")
print(f"WS_02 stage: {propositions_meta.get('stage', 'unknown')}")
print("First proposition:", json.dumps(propositions[0], ensure_ascii=False, indent=2))

## 5.1 Kiểm tra điều kiện trước khi gán nhãn yếu

Checkpoint Bio_ClinicalBERT từ bước 1 và runner bước 3 là hai điều kiện bắt buộc. Ô này **chỉ báo trạng thái**, không tải mô hình hoặc gọi ConceptCLIP/LLM.

In [ ]:
runner_candidates = [
    REPO_DIR / "main_ws_labels.py",
    REPO_DIR / "main_weak_labels.py",
    REPO_DIR / "proposition" / "weak" / "labels.py",
]

teacher_ready = PATHS["teacher_text_model"].is_dir()
runner_path = next((path for path in runner_candidates if path.is_file()), None)

print(f"Text teacher checkpoint: {'READY' if teacher_ready else 'MISSING'}")
print(f"  {PATHS['teacher_text_model']}")
print(f"Stage-03 runner: {runner_path if runner_path else 'NOT IMPLEMENTED'}")
print(f"ConceptCLIP model ID (config): {cfg['image_labeler']['model_id']}")
print(f"LLM provider (config): {cfg['llm_verify']['provider']}")

if not teacher_ready:
    print("\nACTION REQUIRED: Sao chép checkpoint thật của bước 1 lên Drive, sau đó cập nhật teacher_text_model trong configs/ws.yaml.")
if runner_path is None:
    print("\nNEXT IMPLEMENTATION: Cần tạo runner Stage-03 (mask alias, NLI, ConceptCLIP, hợp nhất nhãn, dry-run LLM, labels.npz).")
print("\nKhông có API nào được gọi ở notebook hiện tại.")

## 6. Kiểm tra artifact và chuẩn bị kết quả WS_03

Các file WS_02 được liệt kê ở đây chỉ để xác nhận sự đầy đủ; notebook không tạo hay tải lại chúng. Khi runner WS_03 được triển khai, `labels.npz` và báo cáo mới sẽ được ghi vào thư mục riêng `weak_labels_dir`, không ghi vào `WeakSup/propositions/v1/`.

In [ ]:
stage2_outputs = [
    PATHS["propositions_final"],
    PATHS["propositions_review"],
    PATHS["propositions_summary"],
    PATHS["propositions_meta"],
]

for artifact in stage2_outputs:
    print(f"{'✓' if artifact.is_file() else '✗'} read-only: {artifact}")

print(f"\nReserved WS_03 output directory (not created by this notebook): {PATHS['weak_labels_dir']}")
print("WS_02 artifacts are never downloaded, regenerated, or overwritten here.")

## 7. Báo cáo trạng thái cuối

Chạy ô này sau toàn bộ notebook. Bước 3 chỉ sẵn sàng chạy sau khi đầy đủ artifact WS_02 trên Drive, checkpoint text teacher và runner Stage-03 tồn tại. Phần LLM vẫn phải dừng ở dry-run để thống kê và chờ duyệt trước khi gọi API thật.

In [ ]:
required_items = {
    "SkinCAP train split": PATHS["train_csv"],
    "SkinCAP val split": PATHS["val_csv"],
    "SkinCAP test split": PATHS["test_csv"],
    "SkinCAP images": PATHS["image_root"],
    "WS_02 review CSV (read-only)": PATHS["propositions_review"],
    "WS_02 final propositions (read-only)": PATHS["propositions_final"],
    "WS_02 summary (read-only)": PATHS["propositions_summary"],
    "WS_02 metadata (read-only)": PATHS["propositions_meta"],
    "Text teacher checkpoint": PATHS["teacher_text_model"],
}

status_rows = [
    {"item": name, "path": str(path), "exists": path.exists()}
    for name, path in required_items.items()
]
status = pd.DataFrame(status_rows)
display(status)

ws02_ready = bool(status.loc[status["item"].str.startswith("WS_02"), "exists"].all())
non_teacher_ready = bool(
    status.loc[
        ~status["item"].str.startswith("Text teacher checkpoint"), "exists"
    ].all()
)
ready_for_weak_labels = non_teacher_ready and teacher_ready and runner_path is not None

print(f"\nWS_02 read-only inputs complete: {ws02_ready}")
print("READY FOR STAGE-03 RUN:" if ready_for_weak_labels else "NOT READY FOR STAGE-03 RUN:", ready_for_weak_labels)

if not ready_for_weak_labels:
    print("- Hoàn tất/cập nhật checkpoint text teacher theo đường dẫn thật.")
    print("- Triển khai runner Stage-03 trong branch WS trước khi chạy NLI, ConceptCLIP hoặc LLM.")
    print("- Khi LLM runner đã có: chạy dry-run, xem thống kê, rồi mới phê duyệt API thật.")